In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
np.random.seed(42)


In [2]:
tokens = ["I", "love", "AI"]

seq_len = len(tokens)
d_model = 6        # model dimension
num_heads = 3      # number of attention heads
head_dim = d_model // num_heads

assert d_model % num_heads == 0


In [3]:
E = np.array([
    [1.0, 0.0, 0.2, 0.1, 0.0, 0.3],   # I
    [0.8, 0.6, 0.1, 0.2, 0.1, 0.0],   # love
    [0.0, 1.0, 0.3, 0.0, 0.2, 0.4],   # AI
])

print("Embeddings (E):", E.shape)


Embeddings (E): (3, 6)


In [5]:
# Projection Matrices (Q, K, V, O)

W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)
W_O = np.random.randn(d_model, d_model)

# Linear Projections

Q = E @ W_Q
K = E @ W_K
V = E @ W_V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)



Q shape: (3, 6)
K shape: (3, 6)
V shape: (3, 6)


In [ ]:
def split_heads(x):
    return x.reshape(seq_len, num_heads, head_dim)

# Each token now exists in multiple spaces.

Qh = split_heads(Q)
Kh = split_heads(K)
Vh = split_heads(V)

print("Qh shape:", Qh.shape)


Qh shape: (3, 3, 2)


In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

contexts = []

# -- Each head:

# -- produces its own attention matrix (3 × 3)

# -- learns different relationships

for h in range(num_heads):
    Q_h = Qh[:, h, :]      # (seq_len, head_dim)
    K_h = Kh[:, h, :]
    V_h = Vh[:, h, :]

    scores = Q_h @ K_h.T
    scores /= np.sqrt(head_dim)

    attention = softmax(scores, axis=-1)
    context = attention @ V_h

    contexts.append(context)

    print(f"\nHead {h+1} attention:")
    print(attention)



Head 1 attention:
[[0.324 0.236 0.44 ]
 [0.34  0.272 0.387]
 [0.344 0.27  0.385]]

Head 2 attention:
[[0.178 0.517 0.305]
 [0.158 0.521 0.321]
 [0.495 0.389 0.116]]

Head 3 attention:
[[0.189 0.369 0.442]
 [0.309 0.372 0.319]
 [0.255 0.34  0.405]]


In [9]:
# Concatenate along feature dimension
context_concat = np.concatenate(contexts, axis=-1)

print("\nConcatenated context shape:", context_concat.shape)



Concatenated context shape: (3, 6)


In [ ]:
# WO ​∈ R dmodel​×dmodel​

output = context_concat @ W_O

print("\nFinal output shape:", output.shape)
print("\nFinal output:")
print(output)



Final output shape: (3, 6)

Final output:
[[-1.631 -0.192 -1.734  3.686  2.781  1.138]
 [-1.686 -0.292 -1.877  3.729  2.62   1.268]
 [-2.102  0.197 -2.092  3.643  2.882  1.018]]
